# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
infection_type = 'sec'
reg_model = ''
runs = '-200-'
comment = 'no-cell-var'
d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_nets = []
parameters_nets = []
prim_diff_bias_nets = []
sec_diff_bias_nets = []
cell_series_nets = []
lineage_diff_nets = []

In [3]:
# Load data from second infections
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and infection_type in f]

for f in tqdm(file_list):
    filepath = os.path.join(os.path.join(d, "raw"), f)
    with open(filepath, 'rb') as filename:  
        import_dict = pickle.load(filename)

    mean_prim_diff_bias = []
    std_prim_diff_bias = []
    mean_sec_diff_bias = []
    std_sec_diff_bias = []
    mean_cell_series = []
    std_cell_series = []
    mean_lineage_diff = []
    std_lineage_diff = []
    mean_sim_sum = []
    std_sim_sum = []

    parameters = np.array(import_dict["parameters"])
    prim_diff_bias = np.array(import_dict["prim_diff_bias"])
    sec_diff_bias = np.array(import_dict["sec_diff_bias"])
    cell_series = np.array(import_dict["cell_time_series"])
    lineage_diff = np.array(import_dict["lineage_diff"])
    sim_sum = np.array(import_dict["sumary_stats"])

    virs = np.unique(parameters[:,[4,7]], axis = 0)
    
    for i, vir in enumerate(virs):
        index = (parameters[:,4] == vir[0])*(parameters[:,7] == vir[1])
                
        mean_sim_sum.append(np.mean(sim_sum[index], axis = 0))
        std_sim_sum.append(np.std(sim_sum[index], axis = 0))

        mean_prim_diff_bias.append(np.mean(prim_diff_bias[index], axis = 0))
        std_prim_diff_bias.append(np.std(prim_diff_bias[index], axis = 0))

        mean_sec_diff_bias.append(np.mean(sec_diff_bias[index], axis = 0))
        std_sec_diff_bias.append(np.std(sec_diff_bias[index], axis = 0))
        
        mean_cell_series.append(np.mean(cell_series[index], axis = 0))
        std_cell_series.append(np.std(cell_series[index], axis = 0))
        
        mean_lineage_diff.append(np.mean(lineage_diff[index], axis = 0))
        std_lineage_diff.append(np.std(lineage_diff[index], axis = 0))

    # Save datasets
    ### (1) Summary stats
    np.save(os.path.join(d, "summary_stats","mean",f[:-4]), mean_sim_sum)
    np.save(os.path.join(d, "summary_stats","std",f[:-4]), std_sim_sum)
    ### (2) Differentiation bias
    np.save(os.path.join(d, "prim_diff_bias","mean",f[:-4]), mean_prim_diff_bias)
    np.save(os.path.join(d, "prim_diff_bias","std",f[:-4]), std_prim_diff_bias)
    np.save(os.path.join(d, "sec_diff_bias","mean",f[:-4]), mean_sec_diff_bias)
    np.save(os.path.join(d, "sec_diff_bias","std",f[:-4]), std_sec_diff_bias)
    ### (3) Cell time series
    np.save(os.path.join(d, "cell_time_series","mean",f[:-4]), mean_cell_series)
    np.save(os.path.join(d, "cell_time_series","std",f[:-4]), std_cell_series)
    ### (4) Lineage differentiation
    np.save(os.path.join(d, "lineage_diff","mean",f[:-4]), mean_lineage_diff)
    np.save(os.path.join(d, "lineage_diff","std",f[:-4]), std_lineage_diff)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5708/5708 [56:30<00:00,  1.68it/s]


In [4]:
print(filename)
with open(filepath, 'rb') as filename: 
        import_dict = pickle.load(filename)

<_io.BufferedReader name='/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/0-0-2-0-2-0-200-sec-0-1-1-no-cell-var.pkl'>


In [5]:
# # Check dataset in case of an error
# with open(filepath, 'rb') as filename: 
#     import_dict = pickle.load(filename)

In [6]:
# Save stacked datasets
comment = 'no-cell-var'
d_mean = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/'
d_std = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/'

mean_nets = []
std_nets = []

file_list = [f for f in os.listdir(d_mean) if runs in f and infection_type in f]

for f in tqdm(file_list):
    net_mean = np.load(os.path.join(d_mean, f))
    net_std = np.load(os.path.join(d_std, f))
    mean_nets.append(net_mean)
    std_nets.append(net_std)

mean_data = np.vstack(mean_nets)
std_data = np.vstack(std_nets)

np.save(os.path.join(d, "summary_stats","mean","stacked_data"+runs+"runs"), mean_data)
np.save(os.path.join(d, "summary_stats","std","stacked_data"+runs+"runs"), std_data)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5708/5708 [00:07<00:00, 795.62it/s]
